In [3]:
# 1. Imports and project paths

import os
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19, VGG19_Weights

PROJECT_DIR = Path(r"D:\APNA COLLEGE\projects_big\NST\project")
CONTENT_DIR = PROJECT_DIR / "dataset" / "test2017"
STYLE_DIR = PROJECT_DIR / "dataset" / "style"

assert CONTENT_DIR.exists(), f"Content folder not found: {CONTENT_DIR}"
assert STYLE_DIR.exists(), f"Style folder not found: {STYLE_DIR}"

print("Project paths verified.")

Project paths verified.


In [4]:
# 2. GPU setup

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("CUDA is unavailable; running on CPU.")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU memory: 8.0 GB


In [5]:
# 3. Collect content and style image paths

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def get_images(folder):
    return [
        path for path in Path(folder).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]

content_images = get_images(CONTENT_DIR)
style_images = get_images(STYLE_DIR)

if not content_images:
    raise RuntimeError("No content images found.")
if not style_images:
    raise RuntimeError("No style images found.")

print("Content images:", len(content_images))
print("Style images:", len(style_images))

Content images: 40671
Style images: 8552


In [6]:
# 4. ImageDataset

class ImageDataset(Dataset):
    def __init__(self, image_paths, image_size=286):
        self.image_paths = image_paths
        self.image_size = image_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = ImageOps.fit(
                image,
                (self.image_size, self.image_size),
                method=Image.Resampling.BICUBIC
            )

        image = TF.pil_to_tensor(image).float() / 255.0
        return image

In [7]:
# 5. Create and verify datasets

content_dataset = ImageDataset(content_images)
style_dataset = ImageDataset(style_images)

content_sample = content_dataset[0]
style_sample = style_dataset[0]

print("Content dataset:", len(content_dataset))
print("Style dataset:", len(style_dataset))
print("Content sample:", content_sample.shape, content_sample.dtype)
print("Style sample:", style_sample.shape, style_sample.dtype)
print("Content pixel range:",
      content_sample.min().item(), "to", content_sample.max().item())
print("Style pixel range:",
      style_sample.min().item(), "to", style_sample.max().item())

Content dataset: 40671
Style dataset: 8552
Content sample: torch.Size([3, 286, 286]) torch.float32
Style sample: torch.Size([3, 286, 286]) torch.float32
Content pixel range: 0.0 to 1.0
Style pixel range: 0.0 to 0.9843137264251709


In [8]:
# 6. Create DataLoaders

BATCH_SIZE = 4

content_loader = DataLoader(
    content_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True
)

style_loader = DataLoader(
    style_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True
)

print("Content batches:", len(content_loader))
print("Style batches:", len(style_loader))
print("Batch size:", BATCH_SIZE)

Content batches: 10167
Style batches: 2138
Batch size: 4


In [9]:
# 7. VGG19 encoder

class VGGEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        vgg = vgg19(weights=VGG19_Weights.DEFAULT)
        features = vgg.features

        self.encoder1 = nn.Sequential(*features[:2])
        self.encoder2 = nn.Sequential(*features[2:7])
        self.encoder3 = nn.Sequential(*features[7:12])
        self.encoder4 = nn.Sequential(*features[12:21])

        for parameter in self.parameters():
            parameter.requires_grad = False

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, image):
        image = (image - self.mean) / self.std

        f1 = self.encoder1(image)
        f2 = self.encoder2(f1)
        f3 = self.encoder3(f2)
        f4 = self.encoder4(f3)

        return f1, f2, f3, f4


encoder = VGGEncoder().to(DEVICE)
encoder.eval()

print("VGG19 encoder loaded.")
print("Encoder device:", next(encoder.parameters()).device)

VGG19 encoder loaded.
Encoder device: cuda:0


In [10]:
# 8. Verify encoder feature outputs

content_batch = next(iter(content_loader)).to(DEVICE)

with torch.no_grad():
    f1, f2, f3, f4 = encoder(content_batch)

print("Feature 1:", f1.shape)
print("Feature 2:", f2.shape)
print("Feature 3:", f3.shape)
print("Feature 4:", f4.shape)

Feature 1: torch.Size([4, 64, 286, 286])
Feature 2: torch.Size([4, 128, 143, 143])
Feature 3: torch.Size([4, 256, 71, 71])
Feature 4: torch.Size([4, 512, 35, 35])


In [6]:
# 9. Decoder Architecture

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.decoder = nn.Sequential(
            nn.Conv2d(512,256,3,padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 3, 3, padding=1)
        )
    def forward(self, x):
        return self.decoder(x)

decoder = Decoder().to(DEVICE)

In [12]:
# 10. Adaptive Instance Normalization (AdaIN)

def calc_mean_std(feat, eps=1e-5):

    size = feat.size()

    assert len(size) == 4

    N, C = size[:2]

    feat_var = feat.view(N, C, -1).var(
        dim=2, unbiased=False
    ) + eps

    feat_std = feat_var.sqrt()

    feat_mean = feat.view(N, C, -1).mean(dim=2)

    return feat_mean.view(N, C, 1, 1), feat_std.view(N, C, 1, 1)


def adaptive_instance_normalization(content_feat, style_feat):

    assert content_feat.size()[:2] == style_feat.size()[:2]

    style_mean, style_std = calc_mean_std(style_feat)
    content_mean, content_std = calc_mean_std(content_feat)

    normalized_feat = (
        (content_feat - content_mean) / content_std
    )

    return normalized_feat * style_std + style_mean


print("AdaIN function created successfully!")

AdaIN function created successfully!


In [13]:
# Test AdaIN Transformation

content_batch = next(iter(content_loader)).to(DEVICE)
style_batch = next(iter(style_loader)).to(DEVICE)

encoder.eval()

with torch.no_grad():

    content_features = encoder(content_batch)[-1]
    style_features = encoder(style_batch)[-1]

    adain_features = adaptive_instance_normalization(
        content_features,
        style_features
    )
print("Content features:", content_features.shape)
print("Style features:", style_features.shape)
print("AdaIN features:", adain_features.shape)

print("AdaIN transformation successful!")

Content features: torch.Size([4, 512, 35, 35])
Style features: torch.Size([4, 512, 35, 35])
AdaIN features: torch.Size([4, 512, 35, 35])
AdaIN transformation successful!


In [14]:
# Generate Stylized Image

decoder.eval()
encoder.eval()

with torch.no_grad():

    output = decoder(adain_features)

    output = output.clamp(0,1)

 
print("Generated image shape:", output.shape)
print("Stylized image generated successfully!")

Generated image shape: torch.Size([4, 3, 280, 280])
Stylized image generated successfully!


In [15]:
# 13. Content and Style Loss Functions

def content_loss(generated_features, target_features):

    return nn.functional.mse_loss(
        generated_features, target_features
    )

def style_loss(generated_features, style_features):

    gen_mean, gen_std = calc_mean_std(generated_features)
    style_mean, style_std = calc_mean_std(style_features)

    loss_mean = nn.functional.mse_loss(gen_mean, style_mean)
    loss_std = nn.functional.mse_loss(gen_std, style_std)

    return loss_mean + loss_std

In [16]:
# 14. Optimizer setup

optimizer = torch.optim.Adam(decoder.parameters(), lr=1e-4)
print("Optimizer initialized successfully!")
print("Optimizer: Adam")
print("Learning Rate: 0.0001")

Optimizer initialized successfully!
Optimizer: Adam
Learning Rate: 0.0001


In [17]:
# 15. Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)
print("Learning Rate Scheduler initialized!")
print("Step Size: 10")
print("Gamma: 0.5")

Learning Rate Scheduler initialized!
Step Size: 10
Gamma: 0.5


In [18]:
# 16. Training Function

def train_one_epoch():

    decoder.train()
    encoder.eval()

    total_loss = 0.0
    total_batches = 0

    for content_batch, style_batch in zip(content_loader, style_loader):

        content_batch = content_batch.to(DEVICE)
        style_batch = style_batch.to(DEVICE)

        optimizer.zero_grad()

        with torch.no_grad():
            content_features = encoder(content_batch)[-1]
            style_features = encoder(style_batch)[-1]

            target_features = adaptive_instance_normalization(
                content_features, style_features
            )

            generated_image = decoder(target_features)

            generated_features = encoder(generated_image)[-1]

            loss_c = content_loss(generated_features, target_features)

            loss_s = style_loss(generated_features, style_features)

            loss = loss_c + loss_s

            loss.backward()

            optimizer.step()

            total_loss += loss.item()
            total_batches += 1

        scheduler.step()

        return total_loss / total_batches


print("Training function created successfully!")

Training function created successfully!


In [19]:
# 17. One Batch Training Validation - Gradient Fix

decoder.train()
encoder.eval()

# Make sure decoder is trainable
for param in decoder.parameters():
    param.requires_grad = True

# Load one batch
content_batch = next(iter(content_loader)).to(DEVICE)
style_batch = next(iter(style_loader)).to(DEVICE)

optimizer.zero_grad()

# Extract target features without gradients
with torch.no_grad():

    content_features = encoder(content_batch)[-1]
    style_features = encoder(style_batch)[-1]

    target_features = adaptive_instance_normalization(
        content_features,
        style_features
    )

# Enable gradients for decoder and generated image path
with torch.enable_grad():

    generated_image = decoder(target_features)

    generated_features = encoder(generated_image)[-1]

    loss_c = content_loss(
        generated_features,
        target_features
    )

    loss_s = style_loss(
        generated_features,
        style_features
    )

    loss = loss_c + loss_s

    loss.backward()

optimizer.step()

print("Content Loss:", loss_c.item())
print("Style Loss:", loss_s.item())
print("Total Loss:", loss.item())
print("One batch training successful!")

Content Loss: 14.700615882873535
Style Loss: 11.267179489135742
Total Loss: 25.967796325683594
One batch training successful!


In [20]:
# 18. Controlled Training Loop (100 Batches)

def train_model(max_batches=100):

    decoder.train()
    encoder.eval()

    total_loss = 0.0
    batches_trained = 0

    for batch_idx, (content_batch, style_batch) in enumerate(
        zip(content_loader, style_loader)
    ):

        # Stop after 100 batches
        if batches_trained >= max_batches:
            break

        content_batch = content_batch.to(DEVICE)
        style_batch = style_batch.to(DEVICE)

        optimizer.zero_grad()

        # Extract target features
        with torch.no_grad():

            content_features = encoder(content_batch)[-1]
            style_features = encoder(style_batch)[-1]

            target_features = adaptive_instance_normalization(
                content_features,
                style_features
            )

        # Generate image and calculate loss
        with torch.enable_grad():

            generated_image = decoder(target_features)

            generated_features = encoder(generated_image)[-1]

            loss_c = content_loss(
                generated_features,
                target_features
            )

            loss_s = style_loss(
                generated_features,
                style_features
            )

            loss = loss_c + loss_s

            loss.backward()

        # Update decoder weights
        optimizer.step()

        # Update counters
        total_loss += loss.item()
        batches_trained += 1

        if batches_trained % 10 == 0:

            print(
                f"Batch {batches_trained}/{max_batches} "
                f"| Loss: {loss.item():.4f}"
            )

    # Calculate correct average loss
    average_loss = total_loss / batches_trained

    print("\nTraining completed!")
    print("Batches trained:", batches_trained)
    print("Average Loss:", average_loss)

    return average_loss


# Start training
train_loss = train_model(max_batches=100)

Batch 10/100 | Loss: 17.0383
Batch 20/100 | Loss: 17.9461
Batch 30/100 | Loss: 22.3910
Batch 40/100 | Loss: 9.7103
Batch 50/100 | Loss: 30.3417
Batch 60/100 | Loss: 21.2011
Batch 70/100 | Loss: 18.2112
Batch 80/100 | Loss: 21.1082
Batch 90/100 | Loss: 19.4515
Batch 100/100 | Loss: 62.7596

Training completed!
Batches trained: 100
Average Loss: 20.61576891899109


In [ ]:
# 19. Generate and Visualize Sylized Image

import matplotlib.pyplot as plt

decoder.eval()
encoder.eval()

content_batch = next(iter(content_loader)).to(DEVICE)
style_batch = next(iter(style_loader)).to(DEVICE)

with torch.no_grad():

    content_features = encoder(content_batch)[-1]
    style_features = encoder(style_batch)[-1]

    target_features = adaptive_instance_normalization(
        content_features,
        style_features
    )

    generated_image = decoder(target_features)

content_img = content_batch[0].cpu().permute(1,2,0)
style_img = style_batch[0].cpu().permute(1,2,0)
generated_img = generated_image[0].cpu().permute(1,2,0)

# display image

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(content_img.clamp(0,1))
plt.title("content image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(style_img.clamp(0, 1))
plt.title("Style Image")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(generated_img.clamp(0, 1))
plt.title("Generated Image")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

# ============================================================
# ADAIN NEURAL STYLE TRANSFER
# COMPLETE PROJECT RECOVERY (STEP 1 TO STEP 19)
# GPU: NVIDIA RTX 4060
# ============================================================


# STEP 1: IMPORTS AND PROJECT PATHS

import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image, ImageOps

from torchvision.transforms import functional as TF
from torchvision.models import vgg19, VGG19_Weights

from torch.utils.data import Dataset, DataLoader


PROJECT_DIR = Path(r"D:\APNA COLLEGE\projects_big\NST\project")

CONTENT_DIR = PROJECT_DIR / "dataset" / "test2017"
STYLE_DIR = PROJECT_DIR / "dataset" / "style"

assert CONTENT_DIR.exists(), f"Content folder not found: {CONTENT_DIR}"
assert STYLE_DIR.exists(), f"Style folder not found: {STYLE_DIR}"

print("Project paths verified!")


# STEP 2: GPU SETUP

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if DEVICE.type == "cuda":

    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "GPU Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

    torch.cuda.empty_cache()


# STEP 3: COLLECT IMAGE PATHS

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def get_images(folder):

    return [
        path for path in Path(folder).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]


content_images = get_images(CONTENT_DIR)
style_images = get_images(STYLE_DIR)

if not content_images:
    raise RuntimeError("No content images found!")

if not style_images:
    raise RuntimeError("No style images found!")

print("Content images:", len(content_images))
print("Style images:", len(style_images))


# STEP 4: IMAGE DATASET CLASS

IMAGE_SIZE = 256


class ImageDataset(Dataset):

    def __init__(self, image_paths, image_size=IMAGE_SIZE):

        self.image_paths = image_paths
        self.image_size = image_size

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, index):

        image_path = self.image_paths[index]

        with Image.open(image_path) as image:

            image = image.convert("RGB")

            image = ImageOps.fit(
                image,
                (self.image_size, self.image_size),
                method=Image.Resampling.BICUBIC
            )

        image = TF.pil_to_tensor(image).float() / 255.0

        return image


# STEP 5: CREATE DATASETS

content_dataset = ImageDataset(content_images)
style_dataset = ImageDataset(style_images)

print("Content Dataset:", len(content_dataset))
print("Style Dataset:", len(style_dataset))

print("Content Image Shape:", content_dataset[0].shape)
print("Style Image Shape:", style_dataset[0].shape)


# STEP 6: CREATE DATALOADERS

BATCH_SIZE = 1

content_loader = DataLoader(
    content_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True
)

style_loader = DataLoader(
    style_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True
)

print("Content Batches:", len(content_loader))
print("Style Batches:", len(style_loader))
print("Batch Size:", BATCH_SIZE)


# STEP 7: VGG19 ENCODER

class VGGEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        vgg = vgg19(weights=VGG19_Weights.DEFAULT)

        features = vgg.features

        self.encoder1 = nn.Sequential(*features[:2])
        self.encoder2 = nn.Sequential(*features[2:7])
        self.encoder3 = nn.Sequential(*features[7:12])
        self.encoder4 = nn.Sequential(*features[12:21])

        for parameter in self.parameters():

            parameter.requires_grad = False

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, image):

        image = (image - self.mean) / self.std

        f1 = self.encoder1(image)
        f2 = self.encoder2(f1)
        f3 = self.encoder3(f2)
        f4 = self.encoder4(f3)

        return f1, f2, f3, f4


encoder = VGGEncoder().to(DEVICE)

encoder.eval()

print("VGG19 Encoder Loaded!")
print("Encoder Device:", next(encoder.parameters()).device)


# STEP 8: VERIFY VGG19 FEATURES

content_batch = next(iter(content_loader)).to(DEVICE)

with torch.no_grad():

    f1, f2, f3, f4 = encoder(content_batch)

print("Feature 1:", f1.shape)
print("Feature 2:", f2.shape)
print("Feature 3:", f3.shape)
print("Feature 4:", f4.shape)


# STEP 9: DECODER ARCHITECTURE

class Decoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.decoder = nn.Sequential(

            nn.Conv2d(512, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 3, 3, padding=1)
        )

    def forward(self, x):

        return self.decoder(x)


decoder = Decoder().to(DEVICE)

print("Decoder Initialized!")
print("Decoder Device:", next(decoder.parameters()).device)


# STEP 10: ADAPTIVE INSTANCE NORMALIZATION

def calc_mean_std(feat, eps=1e-5):

    size = feat.size()

    N, C = size[:2]

    feat_var = feat.view(N, C, -1).var(
        dim=2,
        unbiased=False
    ) + eps

    feat_std = feat_var.sqrt()

    feat_mean = feat.view(N, C, -1).mean(dim=2)

    return (
        feat_mean.view(N, C, 1, 1),
        feat_std.view(N, C, 1, 1)
    )


def adaptive_instance_normalization(content_feat, style_feat):

    assert content_feat.size()[:2] == style_feat.size()[:2]

    style_mean, style_std = calc_mean_std(style_feat)

    content_mean, content_std = calc_mean_std(content_feat)

    normalized_feat = (
        (content_feat - content_mean) / content_std
    )

    return normalized_feat * style_std + style_mean


print("AdaIN Function Created!")


# STEP 11: TEST ADAIN TRANSFORMATION

content_batch = next(iter(content_loader)).to(DEVICE)
style_batch = next(iter(style_loader)).to(DEVICE)

with torch.no_grad():

    content_features = encoder(content_batch)[-1]
    style_features = encoder(style_batch)[-1]

    adain_features = adaptive_instance_normalization(
        content_features,
        style_features
    )

print("Content Features:", content_features.shape)
print("Style Features:", style_features.shape)
print("AdaIN Features:", adain_features.shape)


# STEP 12: TEST DECODER

decoder.eval()

with torch.no_grad():

    test_output = decoder(adain_features)

print("Generated Image Shape:", test_output.shape)


# STEP 13: CONTENT AND STYLE LOSS

def content_loss(generated_features, target_features):

    return nn.functional.mse_loss(
        generated_features,
        target_features
    )


def style_loss(generated_features, style_features):

    gen_mean, gen_std = calc_mean_std(generated_features)

    style_mean, style_std = calc_mean_std(style_features)

    loss_mean = nn.functional.mse_loss(
        gen_mean,
        style_mean
    )

    loss_std = nn.functional.mse_loss(
        gen_std,
        style_std
    )

    return loss_mean + loss_std


print("Loss Functions Created!")


# STEP 14: ADAM OPTIMIZER

optimizer = optim.Adam(
    decoder.parameters(),
    lr=1e-4
)

print("Adam Optimizer Created!")


# STEP 15: LEARNING RATE SCHEDULER

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)

print("Learning Rate Scheduler Created!")


# STEP 16: TRAINING FUNCTION

def train_one_epoch(max_batches=100):

    decoder.train()
    encoder.eval()

    total_loss = 0.0
    batches_trained = 0

    for content_batch, style_batch in zip(
        content_loader,
        style_loader
    ):

        if batches_trained >= max_batches:
            break

        content_batch = content_batch.to(DEVICE)
        style_batch = style_batch.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        # Extract target features
        with torch.no_grad():

            content_features = encoder(content_batch)[-1]
            style_features = encoder(style_batch)[-1]

            target_features = adaptive_instance_normalization(
                content_features,
                style_features
            )

        # Generate image and calculate gradients
        generated_image = decoder(target_features)

        generated_features = encoder(generated_image)[-1]

        loss_c = content_loss(
            generated_features,
            target_features
        )

        loss_s = style_loss(
            generated_features,
            style_features
        )

        loss = loss_c + loss_s

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        batches_trained += 1

        if batches_trained % 10 == 0:

            print(
                f"Batch {batches_trained}/{max_batches} "
                f"| Loss: {loss.item():.4f}"
            )

        # Release temporary tensors
        del (
            content_features,
            style_features,
            target_features,
            generated_features,
            generated_image,
            loss_c,
            loss_s,
            loss
        )

    if batches_trained == 0:

        raise RuntimeError("No batches were trained.")

    average_loss = total_loss / batches_trained

    print("\nTraining completed!")
    print("Batches Trained:", batches_trained)
    print("Average Loss:", average_loss)

    return average_loss


# STEP 17: ONE BATCH GRADIENT VALIDATION

decoder.train()
encoder.eval()

content_batch = next(iter(content_loader)).to(DEVICE)
style_batch = next(iter(style_loader)).to(DEVICE)

optimizer.zero_grad(set_to_none=True)

with torch.no_grad():

    content_features = encoder(content_batch)[-1]
    style_features = encoder(style_batch)[-1]

    target_features = adaptive_instance_normalization(
        content_features,
        style_features
    )

generated_image = decoder(target_features)

generated_features = encoder(generated_image)[-1]

loss_c = content_loss(generated_features, target_features)

loss_s = style_loss(generated_features, style_features)

loss = loss_c + loss_s

assert loss.requires_grad, "Gradient graph is missing!"

loss.backward()

optimizer.step()

print("Content Loss:", loss_c.item())
print("Style Loss:", loss_s.item())
print("Total Loss:", loss.item())
print("One Batch Training Successful!")


# STEP 18: CONTROLLED TRAINING (100 BATCHES)

train_loss = train_one_epoch(max_batches=100)

# Save decoder checkpoint
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "adain_decoder_100_batches.pth"

torch.save(
    {
        "decoder_state_dict": decoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "average_loss": train_loss
    },
    checkpoint_path
)

print("Model checkpoint saved:", checkpoint_path)


# STEP 19: VISUALIZE GENERATED IMAGE

decoder.eval()
encoder.eval()

content_batch = next(iter(content_loader)).to(DEVICE)
style_batch = next(iter(style_loader)).to(DEVICE)

with torch.no_grad():

    content_features = encoder(content_batch)[-1]

    style_features = encoder(style_batch)[-1]

    target_features = adaptive_instance_normalization(
        content_features,
        style_features
    )

    generated_image = decoder(target_features)

# Convert tensors into images
content_img = content_batch[0].cpu().permute(1, 2, 0)

style_img = style_batch[0].cpu().permute(1, 2, 0)

generated_img = generated_image[0].cpu().permute(1, 2, 0)

# Display comparison
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(content_img.clamp(0, 1))
plt.title("Content Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(style_img.clamp(0, 1))
plt.title("Style Image")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(generated_img.clamp(0, 1))
plt.title("Generated Image")
plt.axis("off")

plt.tight_layout()
plt.show()

print("AdaIN Visualization Completed!")


Project paths verified!
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.0 GB
Content images: 40671
Style images: 8552
Content Dataset: 40671
Style Dataset: 8552
Content Image Shape: torch.Size([3, 256, 256])
Style Image Shape: torch.Size([3, 256, 256])
Content Batches: 40671
Style Batches: 8552
Batch Size: 1
VGG19 Encoder Loaded!
Encoder Device: cuda:0
Feature 1: torch.Size([1, 64, 256, 256])
Feature 2: torch.Size([1, 128, 128, 128])
Feature 3: torch.Size([1, 256, 64, 64])
Feature 4: torch.Size([1, 512, 32, 32])
Decoder Initialized!
Decoder Device: cuda:0
AdaIN Function Created!
Content Features: torch.Size([1, 512, 32, 32])
Style Features: torch.Size([1, 512, 32, 32])
AdaIN Features: torch.Size([1, 512, 32, 32])
Generated Image Shape: torch.Size([1, 3, 256, 256])
Loss Functions Created!
Adam Optimizer Created!
Learning Rate Scheduler Created!
Content Loss: 21.867626190185547
Style Loss: 17.08646011352539
Total Loss: 38.95408630371094
One Batch Training Successful!
B

In [1]:
# STEP 20: Verify Saved Model Checkpoint

from pathlib import Path

PROJECT_DIR = Path(r"D:\APNA COLLEGE\projects_big\NST\project")

checkpoint_path = (
    PROJECT_DIR / "checkpoints" / "adain_decoder_100_batches.pth"
)

print("Checkpoint Path:", checkpoint_path)

if checkpoint_path.exists():

    print("Checkpoint Found Successfully!")
    print("File Size:", round(checkpoint_path.stat().st_size / 1024**2, 2), "MB")

else:

    print("Checkpoint Not Found!")

Checkpoint Path: D:\APNA COLLEGE\projects_big\NST\project\checkpoints\adain_decoder_100_batches.pth
Checkpoint Found Successfully!
File Size: 26.63 MB


In [3]:
# STEP 21: Recreate Decoder and Load Trained Checkpoint

import torch
import torch.nn as nn
from pathlib import Path

# GPU setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Project path
PROJECT_DIR = Path(
    r"D:\APNA COLLEGE\projects_big\NST\project"
)

checkpoint_path = (
    PROJECT_DIR / "checkpoints" / "adain_decoder_100_batches.pth"
)


# 1. Define Decoder Architecture

class Decoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.decoder = nn.Sequential(

            nn.Conv2d(512, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 3, 3, padding=1)
        )

    def forward(self, x):
        return self.decoder(x)


# 2. Initialize Decoder

decoder = Decoder().to(DEVICE)


# 3. Load Checkpoint

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=True
)


# 4. Restore Trained Weights

decoder.load_state_dict(
    checkpoint["decoder_state_dict"]
)

decoder.eval()


# 5. Verify

print("Trained Decoder Loaded Successfully!")
print("Device:", DEVICE)
print("Training Average Loss:", checkpoint["average_loss"])
print("Checkpoint Size:",
      round(checkpoint_path.stat().st_size / 1024**2, 2), "MB")

Trained Decoder Loaded Successfully!
Device: cuda
Training Average Loss: 26.477837247848512
Checkpoint Size: 26.63 MB


In [4]:
# STEP 22: Check Required Variables

required_variables = [
    "encoder",
    "decoder",
    "content_loader",
    "style_loader",
    "adaptive_instance_normalization",
    "DEVICE"
]

for variable in required_variables:

    print(
        variable,
        "Available" if variable in globals() else "Missing"
    )

encoder Missing
decoder Available
content_loader Missing
style_loader Missing
adaptive_instance_normalization Missing
DEVICE Available


In [ ]:

# STEP 22: RESTORE MISSING COMPONENTS AND GENERATE IMAGE

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image, ImageOps

from torchvision.transforms import functional as TF
from torchvision.models import vgg19, VGG19_Weights
from torch.utils.data import Dataset, DataLoader


# 1. PROJECT PATHS

PROJECT_DIR = Path(
    r"D:\APNA COLLEGE\projects_big\NST\project"
)

CONTENT_DIR = PROJECT_DIR / "dataset" / "test2017"
STYLE_DIR = PROJECT_DIR / "dataset" / "style"

CHECKPOINT_PATH = (
    PROJECT_DIR / "checkpoints" / "adain_decoder_100_batches.pth"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

IMAGE_SIZE = 256
BATCH_SIZE = 1


# 2. RESTORE DATASET CLASS

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def get_images(folder):

    return [
        path for path in Path(folder).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]


class ImageDataset(Dataset):

    def __init__(self, image_paths):

        self.image_paths = image_paths

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, index):

        with Image.open(self.image_paths[index]) as image:

            image = image.convert("RGB")

            image = ImageOps.fit(
                image,
                (IMAGE_SIZE, IMAGE_SIZE),
                method=Image.Resampling.BICUBIC
            )

        return TF.pil_to_tensor(image).float() / 255.0


# 3. RESTORE DATALOADERS

content_images = get_images(CONTENT_DIR)
style_images = get_images(STYLE_DIR)

content_dataset = ImageDataset(content_images)
style_dataset = ImageDataset(style_images)

content_loader = DataLoader(
    content_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

style_loader = DataLoader(
    style_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

print("Content images:", len(content_dataset))
print("Style images:", len(style_dataset))


# 4. RESTORE VGG19 ENCODER

class VGGEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        vgg = vgg19(weights=VGG19_Weights.DEFAULT)

        features = vgg.features

        self.encoder1 = nn.Sequential(*features[:2])
        self.encoder2 = nn.Sequential(*features[2:7])
        self.encoder3 = nn.Sequential(*features[7:12])
        self.encoder4 = nn.Sequential(*features[12:21])

        for parameter in self.parameters():

            parameter.requires_grad = False

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, image):

        image = (image - self.mean) / self.std

        f1 = self.encoder1(image)
        f2 = self.encoder2(f1)
        f3 = self.encoder3(f2)
        f4 = self.encoder4(f3)

        return f1, f2, f3, f4


encoder = VGGEncoder().to(DEVICE)
encoder.eval()

print("VGG19 Encoder Restored!")


# 5. RESTORE ADAIN FUNCTION

def calc_mean_std(feat, eps=1e-5):

    N, C = feat.size()[:2]

    feat_var = feat.view(N, C, -1).var(
        dim=2,
        unbiased=False
    ) + eps

    feat_std = feat_var.sqrt()

    feat_mean = feat.view(N, C, -1).mean(dim=2)

    return (
        feat_mean.view(N, C, 1, 1),
        feat_std.view(N, C, 1, 1)
    )


def adaptive_instance_normalization(content_feat, style_feat):

    style_mean, style_std = calc_mean_std(style_feat)

    content_mean, content_std = calc_mean_std(content_feat)

    normalized_feat = (
        content_feat - content_mean
    ) / content_std

    return normalized_feat * style_std + style_mean


print("AdaIN Function Restored!")


# 6. RESTORE DECODER ARCHITECTURE

class Decoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.decoder = nn.Sequential(

            nn.Conv2d(512, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 3, 3, padding=1)
        )

    def forward(self, x):

        return self.decoder(x)


decoder = Decoder().to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=True
)

decoder.load_state_dict(checkpoint["decoder_state_dict"])
decoder.eval()

print("Trained Decoder Restored!")


# 7. GENERATE STYLIZED IMAGE

content_batch = next(iter(content_loader)).to(DEVICE)
style_batch = next(iter(style_loader)).to(DEVICE)

with torch.no_grad():

    content_features = encoder(content_batch)[-1]
    style_features = encoder(style_batch)[-1]

    target_features = adaptive_instance_normalization(
        content_features,
        style_features
    )

    generated_image = decoder(target_features)


# 8. VISUALIZE RESULTS

content_img = content_batch[0].cpu().permute(1, 2, 0)
style_img = style_batch[0].cpu().permute(1, 2, 0)
generated_img = generated_image[0].cpu().permute(1, 2, 0)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(content_img.clamp(0, 1))
plt.title("Content Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(style_img.clamp(0, 1))
plt.title("Style Image")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(generated_img.clamp(0, 1))
plt.title("Generated Image")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Stylized Image Generated Successfully!")


Content images: 40671
Style images: 8552
VGG19 Encoder Restored!
AdaIN Function Restored!
Trained Decoder Restored!


In [1]:
# STEP 1: Clean GPU Setup

import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Free VRAM:",
        round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
        "GB"
    )

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Free VRAM: 6.94 GB


In [2]:
# STEP 2: Restore VGG19 Encoder

import torch
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights

class VGGEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        vgg = vgg19(weights=VGG19_Weights.DEFAULT)

        features = vgg.features

        self.encoder1 = nn.Sequential(*features[:2])
        self.encoder2 = nn.Sequential(*features[2:7])
        self.encoder3 = nn.Sequential(*features[7:12])
        self.encoder4 = nn.Sequential(*features[12:21])

        for param in self.parameters():
            param.requires_grad = False

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, image):

        image = (image - self.mean) / self.std

        f1 = self.encoder1(image)
        f2 = self.encoder2(f1)
        f3 = self.encoder3(f2)
        f4 = self.encoder4(f3)

        return f1, f2, f3, f4


# Load encoder directly on GPU
encoder = VGGEncoder().to(DEVICE)
encoder.eval()

print("VGG19 Encoder Restored!")
print("Device:", next(encoder.parameters()).device)
print("Trainable Parameters:",
      sum(p.numel() for p in encoder.parameters() if p.requires_grad))

VGG19 Encoder Restored!
Device: cuda:0
Trainable Parameters: 0


In [3]:
# STEP 3: Restore Content and Style DataLoaders

from pathlib import Path
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF
from torch.utils.data import Dataset, DataLoader

PROJECT_DIR = Path(
    r"D:\APNA COLLEGE\projects_big\NST\project"
)

CONTENT_DIR = PROJECT_DIR / "dataset" / "test2017"
STYLE_DIR = PROJECT_DIR / "dataset" / "style"

IMAGE_SIZE = 256
BATCH_SIZE = 1

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def get_images(folder):

    return [
        p for p in Path(folder).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]


class ImageDataset(Dataset):

    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):

        with Image.open(self.image_paths[index]) as image:

            image = image.convert("RGB")

            image = ImageOps.fit(
                image,
                (IMAGE_SIZE, IMAGE_SIZE),
                method=Image.Resampling.BICUBIC
            )

        return TF.pil_to_tensor(image).float() / 255.0


content_images = get_images(CONTENT_DIR)
style_images = get_images(STYLE_DIR)

assert len(content_images) > 0, "Content images not found!"
assert len(style_images) > 0, "Style images not found!"

content_dataset = ImageDataset(content_images)
style_dataset = ImageDataset(style_images)

content_loader = DataLoader(
    content_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

style_loader = DataLoader(
    style_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

print("Content Images:", len(content_dataset))
print("Style Images:", len(style_dataset))
print("Content Batch Size:", content_loader.batch_size)
print("Style Batch Size:", style_loader.batch_size)
print("DataLoaders Restored Successfully!")

Content Images: 40671
Style Images: 8552
Content Batch Size: 1
Style Batch Size: 1
DataLoaders Restored Successfully!


In [4]:
# STEP 4A: Restore AdaIN Function

def calc_mean_std(feat, eps=1e-5):

    N, C = feat.size()[:2]

    feat_var = feat.view(N, C, -1).var(
        dim=2,
        unbiased=False
    ) + eps

    feat_std = feat_var.sqrt()

    feat_mean = feat.view(N, C, -1).mean(dim=2)

    return (
        feat_mean.view(N, C, 1, 1),
        feat_std.view(N, C, 1, 1)
    )


def adaptive_instance_normalization(content_feat, style_feat):

    style_mean, style_std = calc_mean_std(style_feat)

    content_mean, content_std = calc_mean_std(content_feat)

    normalized_feat = (
        content_feat - content_mean
    ) / content_std

    return normalized_feat * style_std + style_mean


print("AdaIN Function Restored Successfully!")

AdaIN Function Restored Successfully!


In [7]:
# STEP 5A: Generate Stylized Image

decoder.eval()
encoder.eval()

content_batch = next(iter(content_loader)).to(DEVICE)
style_batch = next(iter(style_loader)).to(DEVICE)

with torch.no_grad():

    # Extract content and style features
    content_features = encoder(content_batch)[-1]
    style_features = encoder(style_batch)[-1]

    # Apply AdaIN
    target_features = adaptive_instance_normalization(
        content_features,
        style_features
    )

    # Generate stylized image
    generated_image = decoder(target_features)

print("Content Shape:", content_batch.shape)
print("Style Shape:", style_batch.shape)
print("Generated Shape:", generated_image.shape)

print("Image Generation Completed!")

Content Shape: torch.Size([1, 3, 256, 256])
Style Shape: torch.Size([1, 3, 256, 256])
Generated Shape: torch.Size([1, 3, 256, 256])
Image Generation Completed!


In [ ]:
# STEP 6: Display and Save Generated Image

import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_pil_image

content_img = content_batch[0].detach().cpu().clamp(0, 1)
style_img = style_batch[0].detach().cpu().clamp(0, 1)
generated_img = generated_image[0].detach().cpu().clamp(0, 1)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(to_pil_image(content_img))
plt.title("Content Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(to_pil_image(style_img))
plt.title("Style Image")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(to_pil_image(generated_img))
plt.title("Generated Image")
plt.axis("off")

plt.tight_layout()
plt.show()

# Save result
output_dir = PROJECT_DIR / "generated_results"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "adain_generated.png"

to_pil_image(generated_img).save(output_path)

print("Generated image saved:", output_path)

In [2]:
# STEP 2: CPU Checkpoint Loading Test

import torch
import torch.nn as nn
from pathlib import Path

CPU = torch.device("cpu")

class Decoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.decoder = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="nearest"),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 3, 3, padding=1)
        )

    def forward(self, x):
        return self.decoder(x)


checkpoint_path = Path(
    r"D:\APNA COLLEGE\projects_big\NST\project\checkpoints\adain_decoder_100_batches.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=CPU,
    weights_only=True
)

cpu_decoder = Decoder().to(CPU)

cpu_decoder.load_state_dict(checkpoint["decoder_state_dict"])

cpu_decoder.eval()

# Small dummy input for testing
dummy_input = torch.randn(1, 512, 32, 32)

with torch.inference_mode():

    test_output = cpu_decoder(dummy_input)

print("CPU Checkpoint Test Successful!")
print("Input Shape:", dummy_input.shape)
print("Output Shape:", test_output.shape)
print("Device:", test_output.device)

CPU Checkpoint Test Successful!
Input Shape: torch.Size([1, 512, 32, 32])
Output Shape: torch.Size([1, 3, 256, 256])
Device: cpu


In [3]:
# STEP 3: Restore VGG19 Encoder on CPU

import torch
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights

class VGGEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        features = vgg19(
            weights=VGG19_Weights.DEFAULT
        ).features

        self.encoder1 = nn.Sequential(*features[:2])
        self.encoder2 = nn.Sequential(*features[2:7])
        self.encoder3 = nn.Sequential(*features[7:12])
        self.encoder4 = nn.Sequential(*features[12:21])

        for param in self.parameters():
            param.requires_grad = False

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, image):

        image = (image - self.mean) / self.std

        f1 = self.encoder1(image)
        f2 = self.encoder2(f1)
        f3 = self.encoder3(f2)
        f4 = self.encoder4(f3)

        return f1, f2, f3, f4


cpu_encoder = VGGEncoder().to("cpu")
cpu_encoder.eval()

print("CPU VGG19 Encoder Restored!")
print("Device:", next(cpu_encoder.parameters()).device)

CPU VGG19 Encoder Restored!
Device: cpu


In [2]:
# STEP 3: Check Available RAM

import psutil

memory = psutil.virtual_memory()

print("Total RAM:", round(memory.total / 1024**3, 2), "GB")
print("Used RAM:", round(memory.used / 1024**3, 2), "GB")
print("Available RAM:", round(memory.available / 1024**3, 2), "GB")
print("RAM Usage:", memory.percent, "%")

Total RAM: 15.73 GB
Used RAM: 11.38 GB
Available RAM: 4.35 GB
RAM Usage: 72.4 %


In [3]:
# STEP 4: Low Resolution CPU Test

import torch
import gc

gc.collect()

torch.set_num_threads(2)

# Create a small test tensor
test_image = torch.rand(1, 3, 128, 128)

print("Test Image Shape:", test_image.shape)
print("Device:", test_image.device)
print("CPU Test Successful!")

Test Image Shape: torch.Size([1, 3, 128, 128])
Device: cpu
CPU Test Successful!


In [4]:
# STEP 5: VGG19 Low Memory Test

import torch
from torchvision.models import vgg19, VGG19_Weights

torch.set_num_threads(2)

# Load pretrained VGG19
vgg = vgg19(weights=VGG19_Weights.DEFAULT).features

vgg = vgg[:21].to("cpu")
vgg.eval()

# Freeze parameters
for param in vgg.parameters():
    param.requires_grad = False

# Small input image
test_image = torch.rand(1, 3, 128, 128)

# Extract features
with torch.inference_mode():
    features = vgg(test_image)

print("VGG19 Test Successful!")
print("Input Shape:", test_image.shape)
print("Feature Shape:", features.shape)
print("Device:", features.device)

VGG19 Test Successful!
Input Shape: torch.Size([1, 3, 128, 128])
Feature Shape: torch.Size([1, 512, 16, 16])
Device: cpu


In [5]:
# STEP 5: VGG19 Low Memory Test

import torch
from torchvision.models import vgg19, VGG19_Weights

# CPU configuration
DEVICE = torch.device("cpu")
torch.set_num_threads(2)

# Load pretrained VGG19
vgg = vgg19(
    weights=VGG19_Weights.DEFAULT
).features

# Keep required VGG19 layers
vgg = vgg[:21].to(DEVICE)
vgg.eval()

# Freeze model parameters
for param in vgg.parameters():
    param.requires_grad = False

# Create small test image
test_image = torch.rand(1, 3, 128, 128).to(DEVICE)

# Extract features
with torch.inference_mode():
    features = vgg(test_image)

print("VGG19 Test Successful!")
print("Input Shape:", test_image.shape)
print("Feature Shape:", features.shape)
print("Device:", features.device)

VGG19 Test Successful!
Input Shape: torch.Size([1, 3, 128, 128])
Feature Shape: torch.Size([1, 512, 16, 16])
Device: cpu


In [6]:
# STEP 6: Load Content and Style Images

from pathlib import Path
from PIL import Image
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt
import torch

CONTENT_DIR = PROJECT_DIR / "dataset" / "test2017"
STYLE_DIR = PROJECT_DIR / "dataset" / "style"

def load_image(folder):

    extensions = {".jpg", ".jpeg", ".png", ".webp"}

    image_paths = [
        p for p in Path(folder).rglob("*")
        if p.suffix.lower() in extensions
    ]

    if len(image_paths) == 0:
        raise FileNotFoundError(f"No images found in {folder}")

    with Image.open(image_paths[0]) as img:

        img = img.convert("RGB")
        img = img.resize((128, 128))

        tensor = TF.to_tensor(img).unsqueeze(0)

    return tensor, image_paths[0]


# Load one image from each dataset
content_batch, content_path = load_image(CONTENT_DIR)
style_batch, style_path = load_image(STYLE_DIR)

print("Content Image:", content_path.name)
print("Style Image:", style_path.name)

print("Content Shape:", content_batch.shape)
print("Style Shape:", style_batch.shape)


# Visualize images
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(content_batch[0].permute(1, 2, 0))
plt.title("Content Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(style_batch[0].permute(1, 2, 0))
plt.title("Style Image")
plt.axis("off")

plt.tight_layout()
plt.show()

NameError: name 'PROJECT_DIR' is not defined